# Stevens operators and quadratic anisotropy

We model a single spin with $S=1$ at zero magnetic field.
Rank-two Stevens operators provide an alternative description of the
quadratic anisotropy defined by $D$, $E_x$ and $E_y$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from spinfinity import SpinSys, hamil

## Initialize spin

In [ ]:
spin = SpinSys([1])
spin.B = [0.0, 0.0, 0.0]  # T
spin.BTip = [0.0, 0.0, 0.0]  # T

## 1. Stevens representation

We use the Hamiltonian

$$
H_{\mathrm{Stevens}} = B_2^0 O_2^0 + B_2^2 O_2^2,
$$

with

$$
O_2^0 = 3S_z^2-S(S+1)\mathbb{I},
\qquad
O_2^2 = S_x^2-S_y^2.
$$

The axial term separates the $m_z=0$ state from the $m_z=\pm1$ doublet.
The rhombic term mixes and splits the $m_z=\pm1$ states.

`set_Stevens` automatically enables Stevens anisotropy. While enabled,
the quadratic coefficients `Dvector`, `EvectorX` and `EvectorY` are ignored.

In [ ]:
B20 = 0.10  # meV
B22 = 0.03  # meV

# Clear previous Stevens coefficients when rerunning this cell.
spin.Stevens[0].clear()

# Axial anisotropy.
hamil.set_Stevens(spin, i_spin=0, k=2, q=0, value=B20)
hamil.calc_EigEnergies(spin, rebuildH=True)
print("Axial relative energies (meV):", np.round(spin.E_All, 6))

# Add rhombic anisotropy.
hamil.set_Stevens(spin, i_spin=0, k=2, q=2, value=B22)
hamil.calc_EigEnergies(spin, rebuildH=True)

# Save arrays for comparison; continue using the same SpinSys object.
H_stevens = spin.H.copy()
E_stevens_raw = spin.E_sort.copy()
E_stevens = spin.E_All.copy()

print("Stevens Hamiltonian (meV):")
print(np.round(H_stevens.real, 6))
print("Stevens relative energies (meV):", np.round(E_stevens, 6))

## 2. Comparison with D, Ex and Ey

Spinfinity's quadratic representation is

$$
H_{\mathrm{quadratic}} =
D S_z^2 + E_x S_x^2 + E_y S_y^2.
$$

Using $S_x^2+S_y^2+S_z^2=S(S+1)\mathbb{I}$ gives

$$
B_2^0 = \frac{D-(E_x+E_y)/2}{3},
\qquad
B_2^2 = \frac{E_x-E_y}{2}.
$$

The Hamiltonians differ only by a constant:

$$
H_{\mathrm{quadratic}} =
H_{\mathrm{Stevens}} + C\mathbb{I},
\qquad
C = \frac{S(S+1)}{3}(D+E_x+E_y).
$$

Therefore, transition energies and eigenstates are identical.
For the familiar choice $E_x=-E_y=E$, the conversion reduces to
$B_2^0=D/3$ and $B_2^2=E$.

In [ ]:
D = 0.34  # meV
Ex = 0.07  # meV
Ey = 0.01  # meV

B20_from_quadratic = (D - (Ex + Ey) / 2) / 3
B22_from_quadratic = (Ex - Ey) / 2

np.testing.assert_allclose(
    [B20_from_quadratic, B22_from_quadratic],
    [B20, B22],
    rtol=0,
    atol=1e-12,
)

# Use the quadratic coefficients instead of the Stevens coefficients.
spin.use_Stevens = False
spin.Dvector[0] = D
spin.EvectorX[0] = Ex
spin.EvectorY[0] = Ey
hamil.calc_EigEnergies(spin, rebuildH=True)

H_quadratic = spin.H.copy()
E_quadratic_raw = spin.E_sort.copy()
E_quadratic = spin.E_All.copy()

S = spin.Spins[0]
energy_shift = S * (S + 1) * (D + Ex + Ey) / 3

np.testing.assert_allclose(
    H_quadratic,
    H_stevens + energy_shift * np.eye(spin.dimensionOfMatrix),
    rtol=0,
    atol=1e-12,
)
np.testing.assert_allclose(E_quadratic, E_stevens, rtol=0, atol=1e-12)

print(f"B20 = {B20_from_quadratic:.6f} meV")
print(f"B22 = {B22_from_quadratic:.6f} meV")
print(f"Constant energy shift = {energy_shift:.6f} meV")
print("Quadratic relative energies (meV):", np.round(E_quadratic, 6))
print("Both representations give the same energy differences.")

## 3. Comparing the spectra

The raw eigenenergies differ by $C=0.28$ meV.
After subtracting each ground-state energy, the spectra coincide.

The original axial doublet is split by $2|B_2^2|=0.06$ meV.

In [ ]:
levels = np.arange(spin.dimensionOfMatrix)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), constrained_layout=True)

axes[0].plot(levels, E_stevens_raw, "o", label="Stevens", markersize=8)
axes[0].plot(levels, E_quadratic_raw, "x", label="D, Ex, Ey", markersize=8)
axes[0].set_ylabel("Energy (meV)")
axes[0].set_title("Different energy origins")

axes[1].plot(levels, E_stevens, "o", label="Stevens", markersize=8)
axes[1].plot(levels, E_quadratic, "x", label="D, Ex, Ey", markersize=8)
axes[1].set_ylabel(r"$E_n-E_0$ (meV)")
axes[1].set_title("Identical level splittings")

for ax in axes:
    ax.set_xlabel("Sorted eigenstate index")
    ax.set_xticks(levels)
    ax.grid(alpha=0.3)
    ax.legend()

plt.show()